# Build RAG index trên Google Colab

Notebook này thực hiện các bước còn thiếu cho pipeline online:

`Token chunks → E5-Large embeddings trên GPU → Qdrant local index → smoke test retrieval/reranking → zip artifact`

Artifact cuối cùng không chứa API key và không chứa model cache; model sẽ được tải lại khi cần trên local.

In [ ]:
# Cell 1 — Kiểm tra phiên bản Colab
import sys
print(sys.version)

In [ ]:
# Cell 2 — Clone repo và chọn branch chứa pipeline RAG
import os
from pathlib import Path

REPO_URL = "https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git"
BRANCH = "test_feature_ta"
REPO_DIR = Path("/content/Text-Mining---RAG-on-News")

if not REPO_DIR.exists():
    !git clone --depth 1 --branch $BRANCH $REPO_URL $REPO_DIR
else:
    print(f"Repo đã tồn tại: {REPO_DIR}")

%cd $REPO_DIR
print("Repo:", Path.cwd())
!git branch --show-current
%pip install -q -r requirements.txt
%pip install -q huggingface-hub


In [ ]:
# Cell 3 — Kiểm tra GPU và cấu hình artifact
import json
import os
import sys
import torch
from pathlib import Path

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CẢNH BÁO: Colab chưa bật GPU. Hãy chọn Runtime → Change runtime type → T4 GPU.")

os.environ["HF_HOME"] = "/content/huggingface_cache"
os.environ["NEWS_CHUNK_PATH"] = "src/chunking/output/vieonline_news_chunks_token.jsonl"
os.environ["QDRANT_PATH"] = "data/qdrant_news"
os.environ["QDRANT_COLLECTION"] = "news_bge_token"
os.environ["EMBEDDING_MODEL"] = "intfloat/multilingual-e5-large"
os.environ["RERANKER_MODEL"] = "BAAI/bge-reranker-v2-m3"
os.environ["TOP_K_RETRIEVAL"] = "20"
os.environ["TOP_K_CONTEXT"] = "5"
os.environ["RERANK_MIN_SCORE"] = "2.0"
os.environ["RERANK_MIN_MARGIN"] = "2.0"

CHUNK_PATH = Path(os.environ["NEWS_CHUNK_PATH"])
INDEX_PATH = Path(os.environ["QDRANT_PATH"])
print("Chunk path:", CHUNK_PATH)
print("Index path:", INDEX_PATH)

## Chuẩn bị corpus

Các file generated lớn thường không nằm trong Git. Nếu clone repo không có file chunk, cell sau sẽ cho phép upload file `vieonline_news_chunks_token.jsonl` từ máy hoặc Google Drive.

In [ ]:
# Cell 4 — Kiểm tra/copy corpus chunk
import shutil

if not CHUNK_PATH.exists():
    print("Không tìm thấy corpus trong repo. Hãy upload file vieonline_news_chunks_token.jsonl.")
    from google.colab import files
    uploaded = files.upload()
    source = Path(next(iter(uploaded)))
    CHUNK_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, CHUNK_PATH)

print("Corpus size (MB):", round(CHUNK_PATH.stat().st_size / 1024**2, 2))
print("First lines:")
for index, line in enumerate(CHUNK_PATH.open(encoding="utf-8")):
    if index >= 2:
        break
    row = json.loads(line)
    print({key: row.get(key) for key in ["chunk_id", "article_id", "strategy"]})
    print(str(row.get("chunk_text", row.get("text", "")))[:500], "\n")

In [ ]:
# Cell 5 — Thống kê nhanh corpus để debug chunking
import re

sample_rows = []
num_rows = 0
token_counts = []
strategies = {}
with CHUNK_PATH.open(encoding="utf-8") as handle:
    for line in handle:
        if not line.strip():
            continue
        row = json.loads(line)
        num_rows += 1
        if len(sample_rows) < 5:
            sample_rows.append(row)
        text = str(row.get("text") or row.get("chunk_text") or "")
        token_counts.append(len(re.findall(r"\S+", text)))
        strategy = str(row.get("strategy") or row.get("metadata", {}).get("strategy", "unknown"))
        strategies[strategy] = strategies.get(strategy, 0) + 1

print("Chunks:", num_rows)
EXPECTED_CHUNK_COUNT = 21454
assert num_rows == EXPECTED_CHUNK_COUNT, f"Unexpected chunk count: {num_rows}; expected {EXPECTED_CHUNK_COUNT}"
print("Strategies:", strategies)
print("Approx token stats:", {"min": min(token_counts), "max": max(token_counts), "avg": round(sum(token_counts)/len(token_counts), 2)})
display([{"chunk_id": row.get("chunk_id"), "article_id": row.get("article_id"), "preview": str(row.get("text", ""))[:240]} for row in sample_rows])

In [ ]:
# Cell 6 — Build Qdrant index bằng GPU
# Có thể đổi BATCH_SIZE xuống 32 nếu GPU hết VRAM.
BATCH_SIZE = 64

from src.qa_api.pipeline import NewsPipeline

pipeline = NewsPipeline()
try:
    indexed = pipeline.build_index(batch_size=BATCH_SIZE)
    print(f"Indexed {indexed} chunks into {INDEX_PATH}")
    stored_count = pipeline.client.count(collection_name=os.environ["QDRANT_COLLECTION"], exact=True).count
    print("Qdrant stored points:", stored_count)
    assert indexed == EXPECTED_CHUNK_COUNT
    assert stored_count == EXPECTED_CHUNK_COUNT, f"Qdrant count mismatch: {stored_count}"
finally:
    pipeline.close()

print("Index files:")
for path in sorted(INDEX_PATH.rglob("*")):
    if path.is_file():
        print(path.relative_to(INDEX_PATH), round(path.stat().st_size / 1024**2, 2), "MB")

In [ ]:
# Cell 7 — Smoke test semantic retrieval với QA bắt buộc
from src.qa_api.pipeline import NewsPipeline

smoke_questions = [
    {"question": "Ai là đạo diễn của bộ phim truyền hình Đi về phía lửa?", "expected_article_id": "157600"},
    {"question": "Tại sao nước dùng hầm xương có thể ảnh hưởng đến sức khỏe?", "expected_article_id": "211640"},
]
pipeline = NewsPipeline()
try:
    for case in smoke_questions:
        retrieved = pipeline.retrieve(case["question"], limit=int(os.environ["TOP_K_RETRIEVAL"]))
        article_ids = [str(row.get("article_id")) for row in retrieved]
        print("Query:", case["question"])
        print("Top articles:", article_ids[:5])
        print("Top scores:", [round(float(row.get("retrieval_score", 0.0)), 4) for row in retrieved[:5]])
        assert case["expected_article_id"] in article_ids, ("Semantic retrieval failed", case, article_ids[:10])
finally:
    pipeline.close()

In [ ]:
# Cell 8 — Smoke test BGE reranking trên GPU/CPU
pipeline = NewsPipeline()
try:
    for case in smoke_questions:
        retrieved = pipeline.retrieve(case["question"], limit=int(os.environ["TOP_K_RETRIEVAL"]))
        reranked = pipeline.rerank(case["question"], retrieved)[:int(os.environ["TOP_K_CONTEXT"])]
        print("Query:", case["question"])
        display([{"rank": row.get("rank"), "article_id": row.get("article_id"), "retrieval_score": row.get("retrieval_score"), "rerank_score": row.get("rerank_score"), "preview": row.get("text", "")[:300]} for row in reranked])
        assert str(reranked[0].get("article_id")) == case["expected_article_id"], ("BGE reranking failed", case, reranked[0])
finally:
    pipeline.close()

In [ ]:
# Cell 9 — Ghi manifest và zip artifact để tải về
import datetime as dt
import shutil

artifact_root = Path("/content/rag_index_artifact")
if artifact_root.exists():
    shutil.rmtree(artifact_root)
(artifact_root / "data").mkdir(parents=True)
shutil.copytree(INDEX_PATH, artifact_root / "data" / INDEX_PATH.name, ignore=shutil.ignore_patterns(".lock"))

manifest = {
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "chunk_path": str(CHUNK_PATH),
    "chunk_count": num_rows,
    "embedding_model": os.environ["EMBEDDING_MODEL"],
    "reranker_model": os.environ["RERANKER_MODEL"],
    "qdrant_collection": os.environ["QDRANT_COLLECTION"],
    "top_k_retrieval": int(os.environ["TOP_K_RETRIEVAL"]),
    "top_k_context": int(os.environ["TOP_K_CONTEXT"]),
    "note": "Do not copy .lock. Keep the same model and collection settings when loading locally.",
}
(artifact_root / "index_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
(artifact_root / "README.txt").write_text("Copy data/qdrant_news into the project data/ directory. Do not copy .lock.", encoding="utf-8")

zip_path = shutil.make_archive("/content/rag_qdrant_index", "zip", root_dir=artifact_root)
print("Artifact:", zip_path, round(Path(zip_path).stat().st_size / 1024**2, 2), "MB")

In [ ]:
# Cell 10 — Download artifact về máy
from google.colab import files
files.download("/content/rag_qdrant_index.zip")

## Cài artifact trên local

Giải nén nội dung artifact vào thư mục project sao cho có `data/qdrant_news/`. Không copy file `.lock`. Giữ các biến sau trong `.env`:

```text
QDRANT_PATH=data/qdrant_news
QDRANT_COLLECTION=news_bge_token
EMBEDDING_MODEL=intfloat/multilingual-e5-large
RERANKER_MODEL=BAAI/bge-reranker-v2-m3
```

Sau đó chạy `python -m src.qa_api.app`; không cần build lại index.